<a href="https://colab.research.google.com/github/deepinsightai/flash-attention/blob/bahaa-research/blackwell_sdpa_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blackwell Flash SDPA Pilot Benchmark

**Author:** Bahaa Al-Juboori  
**Course:** COMP 9060 Group Project  
**Date:** October 2026

## Purpose

This notebook is an initial compatibility and performance test for the FlashAttention group project.

The goal is to verify that PyTorch's Flash Attention SDPA backend works on the university-provided Google Colab Pro+ Blackwell GPU and to compare it against a naive scaled dot-product attention implementation.

## Environment

- GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
- Compute Capability: 12.0
- GPU Memory: ~95 GB
- PyTorch: 2.11.0+cu130
- CUDA: 13.0
- Data type: BF16

## Pilot Configuration

- Batch size: 1
- Attention heads: 8
- Sequence length: 1024
- Head dimension: 128

## Initial Result

- Naive attention: ~0.0528 ms
- PyTorch Flash SDPA: ~0.0432 ms
- Speedup: ~1.22x

The outputs were also checked for numerical agreement with the naive implementation.

This is only a pilot compatibility test. The final project experiments will evaluate multiple sequence lengths, GPU architectures, attention methods, runtime, memory usage, throughput, and numerical accuracy.

## Timing Method

CUDA events are used instead of CPU wall-clock timing because CUDA operations execute asynchronously. Warm-up iterations are performed before measurement to reduce initialization effects.

In [1]:
!nvidia-smi

Thu Oct 15 00:47:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   32C    P0             49W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Compute capability: (12, 0)
VRAM GB: 94.97076416015625


In [4]:
import torch
import torch.nn.functional as F

print("Flash SDP enabled:", torch.backends.cuda.flash_sdp_enabled())
print("Memory-efficient SDP enabled:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("Math SDP enabled:", torch.backends.cuda.math_sdp_enabled())

Flash SDP enabled: True
Memory-efficient SDP enabled: True
Math SDP enabled: True


In [5]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

device = "cuda"

q = torch.randn(1, 8, 1024, 128, device=device, dtype=torch.bfloat16)
k = torch.randn(1, 8, 1024, 128, device=device, dtype=torch.bfloat16)
v = torch.randn(1, 8, 1024, 128, device=device, dtype=torch.bfloat16)

with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    out = F.scaled_dot_product_attention(q, k, v)

print("Output shape:", out.shape)
print("Success")

Output shape: torch.Size([1, 8, 1024, 128])
Success


In [7]:
max_diff = (out - out_naive).abs().max().item()
mean_diff = (out - out_naive).abs().mean().item()

print("Max absolute difference:", max_diff)
print("Mean absolute difference:", mean_diff)

Max absolute difference: 0.013671875
Mean absolute difference: 0.00022220611572265625


In [6]:
scores = q @ k.transpose(-2, -1)
scores = scores / (128 ** 0.5)
probs = torch.softmax(scores, dim=-1)
out_naive = probs @ v

In [10]:
import time
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

def benchmark_cuda(fn, warmup=20, runs=100):
    for _ in range(warmup):
        fn()

    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    for _ in range(runs):
        fn()

    end.record()
    torch.cuda.synchronize()

    return start.elapsed_time(end) / runs

In [11]:
naive_ms = benchmark_cuda(naive_attention)
flash_ms = benchmark_cuda(flash_attention)

print(f"Naive attention: {naive_ms:.6f} ms")
print(f"Flash SDPA:      {flash_ms:.6f} ms")
print(f"Speedup:         {naive_ms / flash_ms:.2f}x")

Naive attention: 0.052845 ms
Flash SDPA:      0.043203 ms
Speedup:         1.22x
